# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis: one row = one pseudonymized content page** (`content_id`), a single fixed
snapshot — not a page-day. There is no date column to group by; every metric is already
pre-aggregated into one row per page.

**Time window:** each row bundles two fixed windows measured from one export time, not the
calendar dates I'd get from the full release:
- **90-day totals** (`*_90d` columns) — a trailing 90-day window ending at export.
- **30-day comparison** (`*_last_30d` vs `*_prev_30d`) — the most recent 30 days vs. the 30 days
  before that, both inside the 90-day window. `trend_pct` and `trend_direction` (the label source)
  come from comparing these two.

This is a **cross-sectional slice, not a panel** — I get one snapshot per page, not its history.
The full warehouse release (`fact_content_daily_performance`, per `skills/flyrank/flyrank-data`)
is grain `report_date × client × content`, ~17 months (2025-01-27 → 2026-06-30). I'm contracting
against the starter slice here since that's what I can query directly; if I move to the full
release later, this section needs rewriting for the daily grain, not just a bigger row count.

In [ ]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("rows:", len(df))
print("columns:", len(df.columns))
print()
print("content_age_days describes how far back the 90-day window can reach:")
print(df["content_age_days"].describe()[["min", "25%", "50%", "max"]])
print()
print("Every row is >= 90 days old, consistent with a trailing-90-day window:")
print((df["content_age_days"] >= 90).mean(), "fraction with age >= 90")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label / proxy** (never a feature — this is what gets predicted, or what it's computed from):
- `is_declining_label` — the target (built in `01_prepare_features.py`, not in the raw CSV)
- `trend_direction`, `trend_pct` — the label is derived from these; using either as a feature
  would let the model see the answer before predicting it

**Context** (for grouping / splitting / joining only — the model never learns from these):
- `content_id` — pseudonym, one per row, join key
- `client_id` — pseudonym, used for the client-holdout train/test split, never as a feature
- `provider_used`, `model_used` — which LLM wrote the article; operational metadata, not a signal
  about page quality the model should learn from

**Feature** (knowable at scoring time, safe to use — matches `MODEL_NUMERIC_FEATURES` /
`MODEL_CATEGORICAL_FEATURES` in `scripts/ml_utils.py`):
- Numeric: `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, the `log_*_90d`
  traffic totals, `days_with_impressions`, `days_with_sessions`, `content_age_days`,
  `days_since_last_update`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`,
  `ai_traffic_pct`
- Categorical: `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`,
  `word_count_tier`, `impression_tier`, `position_tier`

**Excluded** (each with a reason):
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`,
  `clicks_prev_30d`, `sessions_prev_30d` — these are the raw ingredients of `trend_pct`; even
  though they aren't the label column itself, using them lets the model reconstruct the trend
  directly. Excluded for the same reason as the label columns, not because they're private.
- `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d`, `ai_sessions_90d` —
  raw counts that are already folded into the rate/log features above (`engagement_rate`,
  `scroll_rate`, `ai_traffic_pct`, `log_ai_sessions_90d`); keeping both the raw count and its
  derived rate would double-count the same signal.
- Anything with a name, URL, or domain — none exist in this pseudonymized slice, but this is the
  line I will not cross if I ever touch a less-scrubbed export.

In [ ]:
FEATURES_NUMERIC = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
FEATURES_CATEGORICAL = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
LABEL_SOURCE = ["trend_direction", "trend_pct"]
CONTEXT = ["content_id", "client_id", "provider_used", "model_used"]
EXCLUDED = [
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "pageviews_90d", "users_90d", "engaged_sessions_90d",
    "scroll_events_90d", "ai_sessions_90d",
]

all_named = set(FEATURES_NUMERIC + FEATURES_CATEGORICAL + LABEL_SOURCE + CONTEXT + EXCLUDED)
missing_from_contract = set(df.columns) - all_named
print("columns not yet sorted into a bucket:", sorted(missing_from_contract))
print()
print("(impressions_90d, clicks_90d, sessions_90d are intentionally in this leftover set --")
print(" they're the 90-day totals the log_* features derive from; kept as raw-total context,")
print(" not fed to the model directly since log_impressions_90d etc. already carry the signal.)")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Four claims to check against the actual file, not the docs:
1. **Grain holds** — `content_id` is unique (no page appears twice).
2. **Counts match the promise** — 30,000 rows, 32 clients (per `docs/data-dictionary.md`).
3. **Missingness follows `content_type`**, not random — the dictionary's warning about a blind
   `fillna(0)` needs a number behind it, not just a citation.
4. **Value claim** — `avg_position == 0` really does mean "no data," checked against
   `impressions_90d` rather than assuming the dictionary is right on faith.

In [ ]:
# 1. Grain check: content_id should be unique
dupe_ids = df["content_id"].value_counts()
dupe_ids = dupe_ids[dupe_ids > 1]
print("1. GRAIN — duplicate content_id rows:", len(dupe_ids), "(0 means the grain holds)")

# 2. Counts vs. the documented promise
print("\n2. COUNTS")
print("   rows:", len(df), "-- expected 30,000")
print("   distinct client_id:", df["client_id"].nunique(), "-- expected 32")

# 3. Missingness by content_type -- is it random or patterned?
print("\n3. MISSINGNESS BY content_type (search_volume, word_count)")
missing_by_type = df.groupby("content_type", dropna=False).agg(
    rows=("content_id", "size"),
    search_volume_missing_pct=("search_volume", lambda s: s.isna().mean() * 100),
    word_count_missing_pct=("word_count", lambda s: s.isna().mean() * 100),
)
print(missing_by_type.round(1))

# 4. avg_position == 0 -- is it really "no data", or a real top rank?
print("\n4. WINDOW/VALUE CHECK -- avg_position == 0 rows, cross-checked against impressions_90d")
zero_pos = df[df["avg_position"] == 0]
zero_pos_with_impressions = (zero_pos["impressions_90d"] > 0).sum()
print("   rows with avg_position == 0:", len(zero_pos))
print("   of those, rows WITH impressions_90d > 0:", zero_pos_with_impressions)
print(f"   -> {zero_pos_with_impressions}/{len(zero_pos)} of the avg_position==0 rows DO have")
print("      real impressions. GSC average position is a mean over ranks that start at 1 --")
print("      it can never legitimately compute to exactly 0. Combined with these rows clearly")
print("      having search visibility (impressions > 0), 0 is a sentinel for 'position not")
print("      captured', not a rank -- confirming the dictionary's warning, and ruling out my")
print("      first guess that these rows might just be zero-impression pages with no data at all.")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limits of the starter slice itself (verified above):**
- **No history, one snapshot.** I can say a page is trending down over the last 30 vs. prior 30
  days, but I cannot say *when* the decline started or whether it's accelerating — there's no
  daily series, only two pre-aggregated windows.
- **Missingness is patterned, not random** (confirmed in section 3): keyword-context fields are
  structurally blank for entire `content_type` values, not missing at random. A blind `fillna(0)`
  would silently teach the model "content_type == feedly article" through the back door of a
  fabricated zero, which is why `scripts/01_prepare_features.py` adds `has_*` flags instead.
- **`avg_position == 0` means missing, confirmed by cross-checking impressions** — treating it as
  "ranked #1" (a natural mistake) would corrupt any position-based feature or rule.
- **32 clients only.** Any per-client pattern I find (e.g. "client X's pages decline more") is
  anecdote at this sample size, not something I'd claim generalizes to FlyRank's full client base.

**Limits I inherit from the full warehouse release, if I move to it later** (documented in
`skills/flyrank/flyrank-data/SKILL.md` — I have not queried the gated warehouse myself, so these
are cited claims to verify again once I have access, not numbers I've checked):
- **Unbalanced panel** — per-client history depth varies (some clients ~17 months, others a few)
  — a single global date window would silently favor long-history clients.
- **GSC-only early history** — rows before a client's `ga4_data_start` have GA4 columns zero-filled
  with `ga4_data_available = FALSE`; treating those zeros as "no engagement" instead of "not
  measured yet" would be the same mistake as `avg_position == 0`, in a new place.
- **Query-table window overlap** — `fact_content_query_90d`'s fixed 90-day window overlaps the
  most recent months. If my label lives in that period, `*_last30`-style columns from that table
  would leak the label window into the features; only `*_prev30` columns would be safe.

**What this data can never tell me, in either version:** whether a refresh *causes* traffic to
recover. There's no experiment here — no A/B, no before/after on an actual edit. Every claim I make
downstream has to stay in the range of "this page's demand looks like it's declining," not
"refreshing this page will fix it."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.